# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The unit of analysis is **one content page for one client on one report date**.

Each row contains daily search and analytics performance for a hashed client (`client_hash_id`) and hashed content page (`content_hash_id`) on a specific `report_date`.

The available time window will be verified from the `report_date` field rather than assumed from the dataset description.


The unit of analysis is **one content page for one client on one report date**.

Each row contains daily search and analytics measurements for a hashed client (`client_hash_id`) and hashed content page (`content_hash_id`) on a specific `report_date`.

In the sample inspected, 1,000 rows contained 437 unique content pages across 2 clients. The sample dates ranged from 2025-01-27 to 2025-01-30. These dates describe the inspected sample only; they are not treated as the complete warehouse time window.


In [9]:
import pandas as pd

rows = list(ds.take(1000))
df_sample = pd.DataFrame(rows)

print("Rows sampled:", len(df_sample))
print("Earliest date in sample:", df_sample["report_date"].min())
print("Latest date in sample:", df_sample["report_date"].max())

print("\nUnique clients:", df_sample["client_hash_id"].nunique())
print("Unique content pages:", df_sample["content_hash_id"].nunique())

display(
    df_sample[
        [
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "gsc_data_available",
            "ga4_data_available",
            "gsc_impressions",
            "gsc_clicks",
            "ga4_pageviews",
            "ga4_sessions",
        ]
    ].head(10)
)

Rows sampled: 1000
Earliest date in sample: 2025-01-27
Latest date in sample: 2025-01-30

Unique clients: 2
Unique content pages: 437


,report_date,client_hash_id,content_hash_id,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,ga4_pageviews,ga4_sessions
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,False,30,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,False,5,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,False,1,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,False,6,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,False,5,0,0,0
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e,True,False,21,0,0,0
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,True,False,13,0,0,0
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,True,False,29,0,0,0
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602,True,False,5,0,0,0
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca,True,False,8,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

For the refresh/content opportunity analysis, the candidate predictive features are the observed performance and engagement measurements available for a content page on a given date:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `ga4_pageviews`
* `ga4_sessions`
* `ga4_users`
* `ga4_engaged_sessions`
* `ga4_total_engagement_sec`
* `sessions_organic`
* `sessions_direct`
* `sessions_referral`
* `sessions_social`
* `sessions_paid`
* `sessions_ai`
* `ai_chatgpt`
* `ai_perplexity`
* `ai_gemini`
* `ai_copilot`
* `ai_claude`
* `ai_meta`
* `ai_other`
* `scroll_events`

These fields describe measured performance for the content page on the report date.

### Label

The warehouse does **not contain a direct refresh-outcome label** such as whether a page was refreshed or whether a refresh improved performance.

Therefore, there is **no observed label in this data contract yet**. A future modeling task would need to define a time-based outcome or proxy using historical observations, rather than pretending one of the existing daily measurements is a true refresh-success label.

### Context

These fields provide identity, availability, and data-coverage context:

* `report_date`
* `client_hash_id`
* `content_hash_id`
* `client_has_gsc`
* `client_has_ga4`
* `gsc_data_available`
* `ga4_data_available`

They help determine which observation is being measured and which data sources are available.

### Excluded

No client names, URLs, or private search queries are present in this warehouse extract. The hashed client and content identifiers are retained only to define the observation grain and group records.

I would exclude the identifier fields from model features because they identify entities rather than represent transferable performance signals. I would also exclude availability flags from predictive features unless their use is justified, because they describe data coverage rather than content performance.


In [10]:
# Define the fields in the data contract.

feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events",
]

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

excluded_fields = [
    "client_hash_id",
    "content_hash_id",
]

label_fields = []

print("Feature fields:", len(feature_fields))
print("Label fields:", len(label_fields))
print("Context fields:", len(context_fields))
print("Excluded fields:", len(excluded_fields))

print("\nMissing fields from sample:")
all_declared = set(feature_fields + context_fields)
print(sorted(all_declared - set(df_sample.columns)))

Feature fields: 22
Label fields: 0
Context fields: 7
Excluded fields: 2

Missing fields from sample:
[]


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The checks below verify the data contract using the observed sample. I will verify the proposed grain, row and entity counts, missing values, and the observed date window. Because the dataset is streamed, the date range reported here is explicitly described as the range observed in the sample rather than the complete warehouse range.


In [11]:
# 1. Verify the proposed grain in the sampled data.
grain_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

duplicate_grain_rows = df_sample.duplicated(
    subset=grain_cols
).sum()

print("Duplicate rows at proposed grain:", duplicate_grain_rows)

# 2. Basic counts.
print("\nSample row count:", len(df_sample))
print("Unique clients:", df_sample["client_hash_id"].nunique())
print("Unique content pages:", df_sample["content_hash_id"].nunique())

# 3. Verify the observed sample date window.
print("\nObserved sample date window:")
print("Start:", df_sample["report_date"].min())
print("End:", df_sample["report_date"].max())

# 4. Check missing values in every field.
missing = df_sample.isna().sum()

print("\nFields with missing values:")
display(
    missing[missing > 0]
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

# 5. Check data-source availability.
print("\nData availability:")
print(
    df_sample[
        ["client_has_gsc", "client_has_ga4",
         "gsc_data_available", "ga4_data_available"]
    ].value_counts()
)

Duplicate rows at proposed grain: 0

Sample row count: 1000
Unique clients: 2
Unique content pages: 437

Observed sample date window:
Start: 2025-01-27
End: 2025-01-30

Fields with missing values:


,missing_count



Data availability:
client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available
True            True            True                False                 1000
Name: count, dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has several limits that affect how it can be used for modeling.

First, the inspected data is daily performance data, so it measures what happened on a content page on a given date. It does not directly record whether a content page was refreshed or whether a refresh caused an improvement.

Second, the sample contains GSC and GA4 availability indicators. In the inspected 1,000-row sample, `client_has_ga4` was True while `ga4_data_available` was False for all rows. Therefore, an unavailable analytics measurement should not automatically be interpreted as zero performance.

Third, the warehouse is streamed, so the sample-based date range of 2025-01-27 through 2025-01-30 should not be treated as the complete dataset window.

Fourth, daily observations are repeated for the same client and content page across dates. This means rows are not independent content pages; they are page-day observations. Any future train/test split should respect time to avoid using future information to predict the past.

Finally, there is no direct refresh-success label in the inspected schema. A future scoring model would therefore need a carefully defined future outcome or proxy, constructed only from information available after the prediction point.


In [12]:
# Verify the main data-limit observations from the inspected sample.

print("Sample size:", len(df_sample))

print(
    "\nGA4 client availability:",
    df_sample["client_has_ga4"].value_counts(dropna=False).to_dict()
)

print(
    "GA4 data availability:",
    df_sample["ga4_data_available"].value_counts(dropna=False).to_dict()
)

print(
    "\nGSC client availability:",
    df_sample["client_has_gsc"].value_counts(dropna=False).to_dict()
)

print(
    "GSC data availability:",
    df_sample["gsc_data_available"].value_counts(dropna=False).to_dict()
)

print(
    "\nUnique client-content pairs:",
    df_sample[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Unique client-content-date rows:",
    df_sample[
        ["client_hash_id", "content_hash_id", "report_date"]
    ].drop_duplicates().shape[0]
)

Sample size: 1000

GA4 client availability: {True: 1000}
GA4 data availability: {False: 1000}

GSC client availability: {True: 1000}
GSC data availability: {True: 1000}

Unique client-content pairs: 437
Unique client-content-date rows: 1000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.